In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

torch.cuda.empty_cache()

# model_name = "meta-llama/Llama-3.2-1B"
model_name = "Qwen/Qwen3-0.6B"
model_save_dir = "model/downloaded"
peft_config = LoraConfig(
    r = 8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none" ,
    target_modules=["q_proj","v_proj", "o_proj", "k_proj", "up_proj", "down_proj", "gate_proj"],
)

os.makedirs(model_save_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="cuda",
    cache_dir=model_save_dir
)
peft_model = get_peft_model(model, peft_config)

In [10]:
from evaluation.perplexity import eval_ppl

ppl = eval_ppl(model = model, tokenizer=tokenizer, dataset="wikitext2", seqlen=2048)
print (f"Perplexity before pruning: {ppl}")

evaluating on wikitext2


Token indices sequence length is longer than the specified maximum sequence length for this model (2541000 > 131072). Running this sequence through the model will result in indexing errors


block 0/147
block 50/147
block 100/147
Perplexity before pruning: 19.897984121432994


In [2]:
from pruning.prune import prune_wanda
from pruning.utils import check_sparsity

prune_wanda(model =  peft_model,
            tokenizer = tokenizer,
            sparsity_ratio = 0.3,
            nsamples= 128,
            seqlen= 2048,
            seed = 42)

check_sparsity(model)


loading calibration data


Token indices sequence length is longer than the specified maximum sequence length for this model (528225 > 131072). Running this sequence through the model will result in indexing errors


dataset loading complete
pruning layer 0 name self_attn.q_proj.base_layer
pruning layer 0 name self_attn.q_proj.lora_A.default
pruning layer 0 name self_attn.q_proj.lora_B.default
pruning layer 0 name self_attn.k_proj.base_layer
pruning layer 0 name self_attn.k_proj.lora_A.default
pruning layer 0 name self_attn.k_proj.lora_B.default
pruning layer 0 name self_attn.v_proj.base_layer
pruning layer 0 name self_attn.v_proj.lora_A.default
pruning layer 0 name self_attn.v_proj.lora_B.default
pruning layer 0 name self_attn.o_proj.base_layer
pruning layer 0 name self_attn.o_proj.lora_A.default
pruning layer 0 name self_attn.o_proj.lora_B.default
pruning layer 0 name mlp.gate_proj.base_layer
pruning layer 0 name mlp.gate_proj.lora_A.default
pruning layer 0 name mlp.gate_proj.lora_B.default
pruning layer 0 name mlp.up_proj.base_layer
pruning layer 0 name mlp.up_proj.lora_A.default
pruning layer 0 name mlp.up_proj.lora_B.default
pruning layer 0 name mlp.down_proj.base_layer
pruning layer 0 name ml

0.30413133206745624

In [4]:
from evaluation.perplexity import eval_ppl

ppl = eval_ppl(model = model, tokenizer=tokenizer, dataset="wikitext2", seqlen=2048)
print (f"Perplexity after pruning: {ppl}")

evaluating on wikitext2


block 0/147
block 50/147
block 100/147
Perplexity after pruning: 21.038264827714453


In [ ]:
from data.datasets import get_loaders
from data.utils import prepare_calibration_input

print("Loading data...")
train_data, val_data = get_loaders(name = 'harrison', nsamples=128, seed=42, seqlen=2048, tokenizer=tokenizer)
print("Preparing calibration input...")
inps, outs, attention_mask, position_ids = prepare_calibration_input(peft_model, train_data, device='cuda', max_samples=32)

Loading data...
Preparing calibration input...


In [3]:
from evaluation.perplexity import eval_ppl

ppl = eval_ppl(model = peft_model, tokenizer=tokenizer, dataset="wikitext2", seqlen=2048)
print (f"Perplexity before pruning: {ppl}")

evaluating on wikitext2


Token indices sequence length is longer than the specified maximum sequence length for this model (2541000 > 131072). Running this sequence through the model will result in indexing errors


block 0/147
block 50/147
block 100/147
Perplexity before pruning: 19.897984121432994


In [3]:
from pruning.utils import check_sparsity

foresight_prune(peft_model,
                prune_ratio=0.3,
                mask_lr=0.5,
                nsamples=128,
                seed=42,
                tokenizer=tokenizer)

loading data


Token indices sequence length is longer than the specified maximum sequence length for this model (528225 > 131072). Running this sequence through the model will result in indexing errors


dataset loading complete
preparing calibration input
calibration input prepared
Pruning layer 0, param self_attn.q_proj
This weight is not being recognized self_attn.q_proj
Pruning layer 0, param self_attn.k_proj
Pruning layer 0, param self_attn.v_proj
Pruning layer 0, param self_attn.o_proj
Pruning layer 0, param mlp.gate_proj
Pruning layer 0, param mlp.up_proj
Pruning layer 0, param mlp.down_proj
Pruning layer 1, param self_attn.q_proj
This weight is not being recognized self_attn.q_proj
Pruning layer 1, param self_attn.k_proj
Pruning layer 1, param self_attn.v_proj
Pruning layer 1, param self_attn.o_proj
Pruning layer 1, param mlp.gate_proj
Pruning layer 1, param mlp.up_proj
Pruning layer 1, param mlp.down_proj
Pruning layer 2, param self_attn.q_proj
This weight is not being recognized self_attn.q_proj
Pruning layer 2, param self_attn.k_proj
Pruning layer 2, param self_attn.v_proj
Pruning layer 2, param self_attn.o_proj
Pruning layer 2, param mlp.gate_proj
Pruning layer 2, param mlp

In [4]:
from evaluation.perplexity import eval_ppl

ppl = eval_ppl(model = peft_model, tokenizer=tokenizer, dataset="wikitext2", seqlen=2048)
print (f"Perplexity before pruning: {ppl}")

evaluating on wikitext2


block 0/147
block 50/147
block 100/147
Perplexity before pruning: 21.412954553474748
